# OEIS A007764 — frontier DP on 2×T4

**What this notebook actually does.** It computes `a(n)` exactly, for the largest
`n` that fits a single T4, using the rank-indexed dense frontier DP that is the
one genuinely verified memory result in this repository.

**What it does not do.** It does not compute `a(28)`. That needs ≈1.4 TiB of
device memory even after every valid reduction below; two T4s have 32 GB.
The n=28 figure printed at the end is a measured extrapolation, not a run.

### The results this notebook relies on (each re-derived from scratch)

| Result | Statement | How it is checked here |
|---|---|---|
| Boundary state count | `B(n) = Σ_a M_a·M_{n−a} = M_{n+2} − M_{n+1}` | convolution identity asserted for n≤19 |
| Mid-row peak | `peak(n) = 2·B(n)` | measured occupancy is exactly 1.0000 |
| Bijective ranking | `index = 2·rank(u) + b` onto `[0, 2B(n))` | exhaustive round-trip, n≤8 |
| Dense array | no keys, no hash table, 8→4 B/state | array length == peak live states |

Everything else that phase 1 and phase 2 recorded as a "breakthrough" is either
already contained in the above, or did not survive audit — see
`AUDIT_PHASE1_PHASE2.md`.

## 1. Environment

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi',
                      '--query-gpu=index,name,memory.total',
                      '--format=csv'], capture_output=True, text=True).stdout)
import cupy as cp
ndev = cp.cuda.runtime.getDeviceCount()
print('CuPy', cp.__version__, '| visible GPUs:', ndev)
for d in range(ndev):
    free, total = cp.cuda.Device(d).mem_info
    print(f'  gpu{d}: {free/2**30:.2f} GiB free / {total/2**30:.2f} GiB total')

## 2. Write the sources

The device code is written once and shared verbatim by the CPU reference and the CUDA kernel.

In [ ]:
import os, json, textwrap
os.makedirs('/kaggle/working/src', exist_ok=True)
SOURCES = json.loads(r'''{"a007764_core.py": "\"\"\"Exact frontier DP for OEIS A007764 with rank-indexed dense storage.\n\nThis module is the trusted core: it implements only results that were\nre-derived and verified from scratch (see AUDIT_PHASE1_PHASE2.md).\n\nState space (math/NOTES.md sec.1, independently re-verified):\n  A row-boundary profile is a word of length L = n+1 over\n  {0, '(', ')', M} which factors uniquely as\n\n      (Motzkin word of length a)  M  (Motzkin word of length b),   a + b = n\n\n  because no arc can straddle the M plug.  Hence the number of\n  row-boundary states is\n\n      B(n) = sum_a M_a * M_{n-a} = M_{n+2} - M_{n+1}\n\n  and the mid-row peak is exactly 2*B(n).\n\nSymbols: 0 = EMPTY, 1 = OPEN '(', 2 = CLOSE ')', 3 = MARK.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Dict, List, Tuple\n\nEMPTY, OPEN, CLOSE, MARK = 0, 1, 2, 3\n\nKNOWN_A007764: Dict[int, int] = {\n    1: 2,\n    2: 12,\n    3: 184,\n    4: 8512,\n    5: 1262816,\n    6: 575780564,\n    7: 789360053252,\n    8: 3266598486981642,\n    9: 41044208702632496804,\n    10: 1568758030464750013214100,\n    11: 182413291514248049241470885236,\n    12: 64528039343270018963357185158482118,\n}\n\n\n# --------------------------------------------------------------------------\n# Motzkin tables\n# --------------------------------------------------------------------------\ndef motzkin_numbers(k_max: int) -> List[int]:\n    \"\"\"M[0..k_max] with M_k = #{Motzkin words of length k}.\"\"\"\n    M = [0] * (k_max + 1)\n    M[0] = 1\n    if k_max >= 1:\n        M[1] = 1\n    for k in range(2, k_max + 1):\n        M[k] = M[k - 1] + sum(M[i] * M[k - 2 - i] for i in range(k - 1))\n    return M\n\n\ndef completion_table(k_max: int) -> List[List[int]]:\n    \"\"\"T[rem][d] = #{ways to finish a Motzkin word: rem symbols left, depth d}.\"\"\"\n    T = [[0] * (k_max + 2) for _ in range(k_max + 1)]\n    T[0][0] = 1\n    for rem in range(1, k_max + 1):\n        for d in range(0, k_max + 1):\n            v = T[rem - 1][d]                       # place EMPTY\n            v += T[rem - 1][d + 1]                  # place OPEN\n            if d > 0:\n                v += T[rem - 1][d - 1]              # place CLOSE\n            T[rem][d] = v\n    return T\n\n\n# --------------------------------------------------------------------------\n# Motzkin word ranking  (symbol order EMPTY < OPEN < CLOSE)\n# --------------------------------------------------------------------------\ndef rank_motzkin(word: List[int], T: List[List[int]]) -> int:\n    r, d = 0, 0\n    k = len(word)\n    for i, c in enumerate(word):\n        rem = k - i - 1\n        if c == OPEN:\n            r += T[rem][d]\n            d += 1\n        elif c == CLOSE:\n            r += T[rem][d] + T[rem][d + 1]\n            d -= 1\n        # EMPTY contributes 0\n    assert d == 0\n    return r\n\n\ndef unrank_motzkin(r: int, k: int, T: List[List[int]]) -> List[int]:\n    word: List[int] = []\n    d = 0\n    for i in range(k):\n        rem = k - i - 1\n        c = T[rem][d]\n        if r < c:\n            word.append(EMPTY)\n            continue\n        r -= c\n        c = T[rem][d + 1]\n        if r < c:\n            word.append(OPEN)\n            d += 1\n            continue\n        r -= c\n        word.append(CLOSE)\n        d -= 1\n    assert d == 0 and r == 0\n    return word\n\n\n# --------------------------------------------------------------------------\n# Row-boundary profile ranking\n#   word of length L = n+1  ==  (Motzkin len a) MARK (Motzkin len b),  a+b = n\n#   rank = sum_{a'<a} M_a' * M_{n-a'}  +  rankM(left)*M_b  +  rankM(right)\n# --------------------------------------------------------------------------\nclass ProfileRanker:\n    \"\"\"Bijective ranking of length-(n+1) boundary profiles onto [0, B(n)).\"\"\"\n\n    def __init__(self, n: int) -> None:\n        self.n = n\n        self.L = n + 1\n        self.M = motzkin_numbers(n + 4)\n        self.T = completion_table(n + 4)\n        # offset[a] = number of profiles whose MARK sits strictly left of a\n        self.offset: List[int] = [0] * (n + 2)\n        acc = 0\n        for a in range(n + 1):\n            self.offset[a] = acc\n            acc += self.M[a] * self.M[n - a]\n        self.offset[n + 1] = acc\n        self.size = acc                      # == B(n) == M_{n+2} - M_{n+1}\n\n    def rank(self, word: List[int]) -> int:\n        a = word.index(MARK)\n        b = self.L - 1 - a\n        left, right = word[:a], word[a + 1:]\n        return (self.offset[a]\n                + rank_motzkin(left, self.T) * self.M[b]\n                + rank_motzkin(right, self.T))\n\n    def unrank(self, r: int) -> List[int]:\n        a = 0\n        while r >= self.offset[a + 1]:\n            a += 1\n        r -= self.offset[a]\n        b = self.L - 1 - a\n        ql, qr = divmod(r, self.M[b])\n        return (unrank_motzkin(ql, a, self.T) + [MARK]\n                + unrank_motzkin(qr, b, self.T))\n\n\n# --------------------------------------------------------------------------\n# Frontier (broken-profile) DP, word space.  Reference implementation.\n#\n# Frontier word s[0..W-1], W = n+2.  Before processing vertex (i,j):\n#   s[k], k <  j    : DOWN plug already emitted by (i,k)\n#   s[j]            : LEFT plug entering (i,j)\n#   s[j+1]          : UP   plug entering (i,j)   (DOWN plug of (i-1,j))\n#   s[k], k >  j+1  : UP   plugs for columns k-1\n# After processing (i,j):  s[j] = DOWN plug, s[j+1] = RIGHT plug.\n# End of row: require s[n+1] == EMPTY, then shift one slot right.\n# --------------------------------------------------------------------------\ndef get_slot(s: int, k: int) -> int:\n    return (s >> (2 * k)) & 3\n\n\ndef set_slot(s: int, k: int, v: int) -> int:\n    return (s & ~(3 << (2 * k))) | (v << (2 * k))\n\n\ndef find_partner(s: int, k: int, W: int) -> int:\n    \"\"\"Index of the bracket matching slot k (which must hold OPEN or CLOSE).\"\"\"\n    c = get_slot(s, k)\n    depth = 0\n    if c == OPEN:\n        for t in range(k + 1, W):\n            o = get_slot(s, t)\n            if o == OPEN:\n                depth += 1\n            elif o == CLOSE:\n                if depth == 0:\n                    return t\n                depth -= 1\n    else:\n        for t in range(k - 1, -1, -1):\n            o = get_slot(s, t)\n            if o == CLOSE:\n                depth += 1\n            elif o == OPEN:\n                if depth == 0:\n                    return t\n                depth -= 1\n    raise AssertionError(\"unmatched bracket\")\n\n\ndef successors(s: int, i: int, j: int, n: int) -> List[int]:\n    \"\"\"All frontier words reachable by processing vertex (i,j) from word s.\"\"\"\n    W = n + 2\n    L, U = get_slot(s, j), get_slot(s, j + 1)\n    base = set_slot(set_slot(s, j, EMPTY), j + 1, EMPTY)\n    can_down, can_right = i < n, j < n\n    out: List[int] = []\n\n    if i == 0 and j == 0:                                   # start vertex\n        if L or U:\n            return out\n        if can_down:\n            out.append(set_slot(base, j, MARK))\n        if can_right:\n            out.append(set_slot(base, j + 1, MARK))\n        return out\n\n    if i == n and j == n:                                   # terminal vertex\n        if (L == MARK and U == EMPTY) or (U == MARK and L == EMPTY):\n            out.append(base)\n        return out\n\n    if L == EMPTY and U == EMPTY:\n        out.append(base)                                    # degree 0\n        if can_down and can_right:                          # open a fresh arc\n            out.append(set_slot(set_slot(base, j, OPEN), j + 1, CLOSE))\n        return out\n\n    if L == EMPTY or U == EMPTY:                            # straight / turn\n        v = L if U == EMPTY else U\n        if can_down:\n            out.append(set_slot(base, j, v))\n        if can_right:\n            out.append(set_slot(base, j + 1, v))\n        return out\n\n    # both plugs occupied: the vertex joins two fragments, degree is now 2\n    if L == OPEN and U == CLOSE:\n        return out                                          # would close a cycle\n    if L == MARK:\n        q = find_partner(s, j + 1, W)\n        return [set_slot(base, q, MARK)]\n    if U == MARK:\n        q = find_partner(s, j, W)\n        return [set_slot(base, q, MARK)]\n    a, b = find_partner(s, j, W), find_partner(s, j + 1, W)\n    lo, hi = (a, b) if a < b else (b, a)\n    return [set_slot(set_slot(base, lo, OPEN), hi, CLOSE)]\n\n\ndef a_n_wordspace(n: int, p: int | None = None) -> int:\n    \"\"\"Exact (or mod p) a(n) via dictionary-based frontier DP.\"\"\"\n    W = n + 2\n    full = (1 << (2 * W)) - 1\n    layer: Dict[int, int] = {0: 1}\n    for i in range(n + 1):\n        for j in range(n + 1):\n            nxt: Dict[int, int] = {}\n            for s, v in layer.items():\n                for t in successors(s, i, j, n):\n                    w = nxt.get(t, 0) + v\n                    nxt[t] = w % p if p else w\n            layer = nxt\n        shifted: Dict[int, int] = {}\n        for s, v in layer.items():\n            if get_slot(s, n + 1) != EMPTY:\n                continue\n            t = (s << 2) & full\n            w = shifted.get(t, 0) + v\n            shifted[t] = w % p if p else w\n        layer = shifted\n    return layer.get(0, 0)\n\n\n# --------------------------------------------------------------------------\n# Dense rank-indexed DP  (math/NOTES.md sec.2, the genuine \"A-class\" result)\n#\n# After processing vertex (i,j) the two plugs at slots (j, j+1) can only be\n#     (0,0)      (x,0)      (0,x)      ( '(' , ')' )\n# so contracting those two slots into one gives a length-(n+1) boundary\n# profile u plus one distinguishing bit b, and\n#     index = 2 * rank(u) + b\n# is a bijection onto [0, 2*B(n)).  No hash table, no keys, 100% occupancy.\n# --------------------------------------------------------------------------\ndef word_to_list(s: int, W: int) -> List[int]:\n    return [get_slot(s, k) for k in range(W)]\n\n\ndef list_to_word(xs: List[int]) -> int:\n    s = 0\n    for k, v in enumerate(xs):\n        s |= v << (2 * k)\n    return s\n\n\ndef contract(xs: List[int], j: int) -> Tuple[List[int], int]:\n    \"\"\"Fold slots (j, j+1) of a length-(n+2) word into one slot.\"\"\"\n    lo, hi = xs[j], xs[j + 1]\n    if lo == EMPTY and hi == EMPTY:\n        val, b = EMPTY, 0\n    elif lo == OPEN and hi == CLOSE:\n        val, b = EMPTY, 1\n    elif hi == EMPTY:\n        val, b = lo, 0\n    elif lo == EMPTY:\n        val, b = hi, 1\n    else:\n        raise AssertionError(f\"illegal plug pair {(lo, hi)} at j={j}\")\n    return xs[:j] + [val] + xs[j + 2:], b\n\n\ndef expand(u: List[int], b: int, j: int) -> List[int]:\n    \"\"\"Inverse of contract().\"\"\"\n    val = u[j]\n    if val == EMPTY:\n        pair = (EMPTY, EMPTY) if b == 0 else (OPEN, CLOSE)\n    else:\n        pair = (val, EMPTY) if b == 0 else (EMPTY, val)\n    return u[:j] + [pair[0], pair[1]] + u[j + 1:]\n\n\ndef a_n_dense(n: int, p: int | None = None, report: bool = False) -> int:\n    \"\"\"Exact (or mod p) a(n) on a dense rank-indexed array of length 2*B(n).\n\n    Two index spaces alternate:\n      * row boundary  -> profile rank in [0, B(n))\n      * mid row after vertex (i,j) -> 2*rank(contract_j(s)) + b in [0, 2*B(n))\n    \"\"\"\n    P = ProfileRanker(n)\n    W, size = n + 2, 2 * P.size\n    peak = 0\n\n    def word_before(idx, j, from_boundary):\n        if from_boundary:\n            return [EMPTY] + P.unrank(idx)            # word before (i,0)\n        r, b = divmod(idx, 2)\n        return expand(P.unrank(r), b, j - 1)          # word before (i,j)\n\n    def step(cur, i, j, from_boundary):\n        nxt = {}\n        for idx, v in cur.items():\n            xs = word_before(idx, j, from_boundary)\n            for t in successors(list_to_word(xs), i, j, n):\n                u2, b2 = contract(word_to_list(t, W), j)\n                k = 2 * P.rank(u2) + b2\n                w = nxt.get(k, 0) + v\n                nxt[k] = w % p if p else w\n        return nxt\n\n    def terminal(cur, from_boundary):\n        \"\"\"Vertex (n,n): the MARK is consumed and the frontier empties out.\"\"\"\n        total = 0\n        for idx, v in cur.items():\n            xs = word_before(idx, n, from_boundary)\n            for t in successors(list_to_word(xs), n, n, n):\n                if t == 0:\n                    total += v\n        return (total % p) if p else total\n\n    # seed: process the start vertex (0,0) from the empty frontier\n    cur = {}\n    for t in successors(0, 0, 0, n):\n        u, b = contract(word_to_list(t, W), 0)\n        k = 2 * P.rank(u) + b\n        cur[k] = (cur.get(k, 0) + 1) % p if p else cur.get(k, 0) + 1\n    peak = max(peak, len(cur))\n\n    for i in range(n + 1):\n        j0 = 1 if i == 0 else 0\n        for j in range(j0, n + 1):\n            if i == n and j == n:\n                answer = terminal(cur, from_boundary=(j == 0))\n                if report:\n                    print(f\"    n={n:2d}: array=2*B(n)={size:<14,} peak_live={peak:<14,} \"\n                          f\"occupancy={peak / size:.4f}\")\n                return answer\n            cur = step(cur, i, j, from_boundary=(j == 0))\n            peak = max(peak, len(cur))\n        # row end: contraction is at (n, n+1) and slot n+1 is always EMPTY,\n        # so b == 0 and the next boundary profile rank is exactly idx >> 1\n        cur = {idx >> 1: v for idx, v in cur.items() if not (idx & 1)}\n\n    if report:\n        print(f\"    n={n:2d}: array=2*B(n)={size:<12,} peak_live={peak:<12,} \"\n              f\"occupancy={peak / size:.4f}\")\n    return (sum(cur.values()) % p) if p else sum(cur.values())\n", "a007764_kernel.h": "/* a007764_kernel.h -- frontier DP core shared verbatim by the CPU reference\n * and the CUDA kernel.  Compiles as plain C99 and as CUDA device code.\n *\n * Word encoding: 2 bits per frontier slot inside a u64.\n *   0 EMPTY, 1 OPEN '(', 2 CLOSE ')', 3 MARK\n * Frontier width W = n + 2, so n <= 30 fits a u64.\n *\n * Index spaces (math/NOTES.md sec.2):\n *   boundary : rank(u) in [0, B(n)),  u a length-(n+1) profile\n *   mid row  : 2*rank(u) + b in [0, 2*B(n)) after contracting slots (j, j+1)\n */\n#ifndef A007764_KERNEL_H\n#define A007764_KERNEL_H\n\n#ifdef __CUDACC__\n#define DEVFN __device__ __forceinline__\n#else\n#define DEVFN static inline\n#include <stdint.h>\n#endif\n\ntypedef unsigned long long u64;\ntypedef unsigned int u32;\n\n#define A_EMPTY 0u\n#define A_OPEN  1u\n#define A_CLOSE 2u\n#define A_MARK  3u\n\n/* Tables shared by every thread.  Tstride = n + 6. */\ntypedef struct {\n    const u64 *T;      /* T[rem * Tstride + d] : Motzkin completions        */\n    const u64 *M;      /* M[k] : Motzkin numbers                            */\n    const u64 *off;    /* off[a] : profile-rank offset for MARK at slot a   */\n    int n;\n    int Tstride;\n} Tables;\n\nDEVFN u32 slot_get(u64 s, int k)          { return (u32)((s >> (2 * k)) & 3ull); }\nDEVFN u64 slot_set(u64 s, int k, u32 v)   { return (s & ~(3ull << (2 * k))) | ((u64)v << (2 * k)); }\nDEVFN u64 lowmask(int k)                  { return (k >= 32) ? ~0ull : ((1ull << (2 * k)) - 1ull); }\n\n/* ---- Motzkin word ranking (symbol order EMPTY < OPEN < CLOSE) ---------- */\nDEVFN u64 motzkin_unrank(u64 r, int k, const Tables *tb)\n{\n    u64 w = 0; int d = 0;\n    for (int i = 0; i < k; i++) {\n        int rem = k - i - 1;\n        u64 c = tb->T[rem * tb->Tstride + d];\n        if (r < c) { continue; }                      /* EMPTY */\n        r -= c;\n        c = tb->T[rem * tb->Tstride + d + 1];\n        if (r < c) { w |= (u64)A_OPEN << (2 * i); d++; continue; }\n        r -= c;\n        w |= (u64)A_CLOSE << (2 * i); d--;\n    }\n    return w;\n}\n\nDEVFN u64 motzkin_rank(u64 w, int k, const Tables *tb)\n{\n    u64 r = 0; int d = 0;\n    for (int i = 0; i < k; i++) {\n        int rem = k - i - 1;\n        u32 c = slot_get(w, i);\n        if (c == A_OPEN) {\n            r += tb->T[rem * tb->Tstride + d];\n            d++;\n        } else if (c == A_CLOSE) {\n            r += tb->T[rem * tb->Tstride + d] + tb->T[rem * tb->Tstride + d + 1];\n            d--;\n        }\n    }\n    return r;\n}\n\n/* ---- boundary profile ranking : (Motzkin a) MARK (Motzkin b), a+b = n --- */\nDEVFN u64 profile_unrank(u64 r, const Tables *tb)\n{\n    int n = tb->n, a = 0;\n    while (r >= tb->off[a + 1]) a++;\n    r -= tb->off[a];\n    int b = n - a;\n    u64 Mb = tb->M[b];\n    u64 ql = r / Mb, qr = r - ql * Mb;\n    u64 left  = motzkin_unrank(ql, a, tb);\n    u64 right = motzkin_unrank(qr, b, tb);\n    return left | ((u64)A_MARK << (2 * a)) | (right << (2 * (a + 1)));\n}\n\nDEVFN u64 profile_rank(u64 w, const Tables *tb)\n{\n    int n = tb->n, a = 0;\n    while (slot_get(w, a) != A_MARK) a++;\n    int b = n - a;\n    u64 left  = w & lowmask(a);\n    u64 right = w >> (2 * (a + 1));\n    return tb->off[a] + motzkin_rank(left, a, tb) * tb->M[b]\n                      + motzkin_rank(right, b, tb);\n}\n\n/* ---- contract / expand of the two plug slots (j, j+1) ------------------ */\nDEVFN u64 word_expand(u64 u, u32 b, int j)\n{\n    u32 val = slot_get(u, j), lo, hi;\n    if (val == A_EMPTY) { lo = b ? A_OPEN : A_EMPTY; hi = b ? A_CLOSE : A_EMPTY; }\n    else                { lo = b ? A_EMPTY : val;    hi = b ? val : A_EMPTY;     }\n    return (u & lowmask(j))\n         | ((u64)lo << (2 * j)) | ((u64)hi << (2 * j + 2))\n         | ((u >> (2 * (j + 1))) << (2 * (j + 2)));\n}\n\n/* returns 0 on success and writes the contracted word + bit */\nDEVFN int word_contract(u64 s, int j, u64 *u_out, u32 *b_out)\n{\n    u32 lo = slot_get(s, j), hi = slot_get(s, j + 1), val, b;\n    if (lo == A_EMPTY && hi == A_EMPTY)      { val = A_EMPTY; b = 0; }\n    else if (lo == A_OPEN && hi == A_CLOSE)  { val = A_EMPTY; b = 1; }\n    else if (hi == A_EMPTY)                  { val = lo;      b = 0; }\n    else if (lo == A_EMPTY)                  { val = hi;      b = 1; }\n    else return -1;\n    *u_out = (s & lowmask(j)) | ((u64)val << (2 * j))\n           | ((s >> (2 * (j + 2))) << (2 * (j + 1)));\n    *b_out = b;\n    return 0;\n}\n\n/* ---- bracket partner --------------------------------------------------- */\nDEVFN int find_partner(u64 s, int k, int W)\n{\n    int depth = 0;\n    if (slot_get(s, k) == A_OPEN) {\n        for (int t = k + 1; t < W; t++) {\n            u32 c = slot_get(s, t);\n            if (c == A_OPEN) depth++;\n            else if (c == A_CLOSE) { if (!depth) return t; depth--; }\n        }\n    } else {\n        for (int t = k - 1; t >= 0; t--) {\n            u32 c = slot_get(s, t);\n            if (c == A_CLOSE) depth++;\n            else if (c == A_OPEN) { if (!depth) return t; depth--; }\n        }\n    }\n    return -1;\n}\n\n/* ---- one vertex transition; writes up to two successor words ----------- */\nDEVFN int cell_successors(u64 s, int i, int j, int n, u64 out[2])\n{\n    int W = n + 2;\n    u32 L = slot_get(s, j), U = slot_get(s, j + 1);\n    u64 base = slot_set(slot_set(s, j, A_EMPTY), j + 1, A_EMPTY);\n    int can_down = (i < n), can_right = (j < n), c = 0;\n\n    if (i == 0 && j == 0) {\n        if (L || U) return 0;\n        if (can_down)  out[c++] = slot_set(base, j, A_MARK);\n        if (can_right) out[c++] = slot_set(base, j + 1, A_MARK);\n        return c;\n    }\n    if (i == n && j == n) {\n        if ((L == A_MARK && U == A_EMPTY) || (U == A_MARK && L == A_EMPTY))\n            out[c++] = base;\n        return c;\n    }\n    if (L == A_EMPTY && U == A_EMPTY) {\n        out[c++] = base;\n        if (can_down && can_right)\n            out[c++] = slot_set(slot_set(base, j, A_OPEN), j + 1, A_CLOSE);\n        return c;\n    }\n    if (L == A_EMPTY || U == A_EMPTY) {\n        u32 v = (U == A_EMPTY) ? L : U;\n        if (can_down)  out[c++] = slot_set(base, j, v);\n        if (can_right) out[c++] = slot_set(base, j + 1, v);\n        return c;\n    }\n    if (L == A_OPEN && U == A_CLOSE) return 0;          /* closes a cycle */\n    if (L == A_MARK) { out[c++] = slot_set(base, find_partner(s, j + 1, W), A_MARK); return c; }\n    if (U == A_MARK) { out[c++] = slot_set(base, find_partner(s, j,     W), A_MARK); return c; }\n    {\n        int a = find_partner(s, j, W), b2 = find_partner(s, j + 1, W);\n        int lo = a < b2 ? a : b2, hi = a < b2 ? b2 : a;\n        out[c++] = slot_set(slot_set(base, lo, A_OPEN), hi, A_CLOSE);\n    }\n    return c;\n}\n\n/* ---- rebuild the frontier word standing just before vertex (i,j) ------- */\nDEVFN u64 word_before(u64 idx, int j, int from_boundary, const Tables *tb)\n{\n    if (from_boundary) return profile_unrank(idx, tb) << 2;   /* prepend EMPTY */\n    u64 r = idx >> 1; u32 b = (u32)(idx & 1ull);\n    return word_expand(profile_unrank(r, tb), b, j - 1);\n}\n\n#endif /* A007764_KERNEL_H */\n", "a007764_cuda.cu": "/* CUDA kernels for A007764.  The device functions above this point are\n * a007764_kernel.h verbatim -- the exact code validated on CPU against the\n * twelve known OEIS terms. */\n\nextern \"C\" {\n\n__device__ __forceinline__ void atomic_add_mod(u32 *addr, u32 v, u32 p)\n{\n    u32 old = *addr, assumed;\n    do {\n        assumed = old;\n        u32 nv = assumed + v;\n        if (nv >= p) nv -= p;\n        old = atomicCAS(addr, assumed, nv);\n    } while (assumed != old);\n}\n\n/* Cooperative load of the (tiny) Motzkin tables into shared memory. */\n__device__ __forceinline__ void load_tables(\n        Tables *tb, u64 *smem, const u64 *T, const u64 *M, const u64 *off,\n        int n, int Tstride)\n{\n    int Trows = n + 5, nT = Trows * Tstride, nM = n + 5, nO = n + 2;\n    u64 *sT = smem, *sM = smem + nT, *sO = smem + nT + nM;\n    for (int t = threadIdx.x; t < nT; t += blockDim.x) sT[t] = T[t];\n    for (int t = threadIdx.x; t < nM; t += blockDim.x) sM[t] = M[t];\n    for (int t = threadIdx.x; t < nO; t += blockDim.x) sO[t] = off[t];\n    __syncthreads();\n    tb->T = sT; tb->M = sM; tb->off = sO; tb->n = n; tb->Tstride = Tstride;\n}\n\n/* One vertex of the sweep: cur (size_in) -> nxt (2*B(n)). */\n__global__ void dp_step(\n        const u32 *__restrict__ cur, u32 *nxt,\n        unsigned long long size_in, int i, int j, int n, u32 p,\n        int from_boundary,\n        const u64 *__restrict__ T, const u64 *__restrict__ M,\n        const u64 *__restrict__ off, int Tstride)\n{\n    extern __shared__ u64 smem[];\n    Tables tb;\n    load_tables(&tb, smem, T, M, off, n, Tstride);\n\n    u64 stride = (u64)blockDim.x * gridDim.x;\n    for (u64 idx = (u64)blockIdx.x * blockDim.x + threadIdx.x;\n         idx < size_in; idx += stride) {\n        u32 v = cur[idx];\n        if (!v) continue;\n        u64 s = word_before(idx, j, from_boundary, &tb);\n        u64 out[2];\n        int c = cell_successors(s, i, j, n, out);\n        for (int t = 0; t < c; t++) {\n            u64 u; u32 b;\n            if (word_contract(out[t], j, &u, &b)) continue;   /* unreachable */\n            atomic_add_mod(&nxt[2 * profile_rank(u, &tb) + b], v, p);\n        }\n    }\n}\n\n/* End of row: the contraction sits at (n, n+1) with bit 0, so the next\n * boundary rank is exactly idx >> 1.  Pure gather, no atomics. */\n__global__ void row_end(const u32 *__restrict__ cur, u32 *__restrict__ out,\n                        unsigned long long B)\n{\n    u64 stride = (u64)blockDim.x * gridDim.x;\n    for (u64 r = (u64)blockIdx.x * blockDim.x + threadIdx.x; r < B; r += stride)\n        out[r] = cur[2 * r];\n}\n\n/* Terminal vertex (n,n): the MARK is consumed and the frontier must empty. */\n__global__ void terminal_sum(\n        const u32 *__restrict__ cur, unsigned long long *acc,\n        unsigned long long size_in, int n, int from_boundary,\n        const u64 *__restrict__ T, const u64 *__restrict__ M,\n        const u64 *__restrict__ off, int Tstride)\n{\n    extern __shared__ u64 smem[];\n    Tables tb;\n    load_tables(&tb, smem, T, M, off, n, Tstride);\n\n    unsigned long long local = 0;\n    u64 stride = (u64)blockDim.x * gridDim.x;\n    for (u64 idx = (u64)blockIdx.x * blockDim.x + threadIdx.x;\n         idx < size_in; idx += stride) {\n        u32 v = cur[idx];\n        if (!v) continue;\n        u64 s = word_before(idx, n, from_boundary, &tb);\n        u64 out[2];\n        int c = cell_successors(s, n, n, n, out);\n        for (int t = 0; t < c; t++)\n            if (out[t] == 0ull) local += v;\n    }\n    if (local) atomicAdd(acc, local);\n}\n\n}  /* extern \"C\" */\n", "a007764_gpu.py": "\"\"\"Dual-T4 GPU driver for the A007764 frontier DP.\n\nStorage is the rank-indexed dense array of NOTES.md sec.2: length exactly\n2*B(n) = 2*(M_{n+2} - M_{n+1}), 100% occupied, no keys and no hash table.\nOne CRT prime is one independent full sweep, so primes are handed out to the\navailable GPUs round-robin.\n\nThe device code is a007764_kernel.h (validated on CPU against the twelve known\nOEIS terms) followed by a007764_cuda.cu.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nimport threading\nimport time\nfrom typing import Dict, List, Sequence, Tuple\n\nimport numpy as np\n\nfrom a007764_core import KNOWN_A007764, motzkin_numbers, completion_table\n\n# Primes below 2^31 so that (a + b) stays inside uint32 in the modular atomic.\nCRT_PRIMES_31BIT: List[int] = [\n    2147483647, 2147483629, 2147483587, 2147483579, 2147483563, 2147483549,\n    2147483543, 2147483497, 2147483489, 2147483477, 2147483423, 2147483399,\n    2147483353, 2147483323, 2147483269, 2147483249, 2147483237, 2147483179,\n    2147483171, 2147483137, 2147483123, 2147483077, 2147483069, 2147483059,\n    2147483053, 2147483033, 2147483029, 2147482951, 2147482949, 2147482943,\n    2147482937, 2147482921, 2147482877, 2147482873, 2147482867, 2147482859,\n    2147482819, 2147482817, 2147482811, 2147482801, 2147482763, 2147482739,\n    2147482697, 2147482693, 2147482681, 2147482663, 2147482661, 2147482649,\n]\n\n\ndef _read(path: str) -> str:\n    here = os.path.dirname(os.path.abspath(__file__))\n    with open(os.path.join(here, path)) as f:\n        return f.read()\n\n\ndef cuda_source() -> str:\n    \"\"\"Device header + kernels, concatenated exactly as compiled.\"\"\"\n    return _read(\"a007764_kernel.h\") + \"\\n\" + _read(\"a007764_cuda.cu\")\n\n\n# --------------------------------------------------------------------------\n# host-side tables\n# --------------------------------------------------------------------------\ndef build_tables(n: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, int, int]:\n    kmax, Tstride, Trows = n + 4, n + 6, n + 5\n    T = completion_table(kmax)\n    M = motzkin_numbers(kmax)\n    Tflat = np.zeros(Trows * Tstride, dtype=np.uint64)\n    for rem in range(kmax + 1):\n        for d in range(kmax + 2):\n            Tflat[rem * Tstride + d] = T[rem][d]\n    Marr = np.zeros(n + 5, dtype=np.uint64)\n    for k in range(kmax + 1):\n        Marr[k] = M[k]\n    off = np.zeros(n + 2, dtype=np.uint64)\n    acc = 0\n    for a in range(n + 1):\n        off[a] = acc\n        acc += M[a] * M[n - a]\n    off[n + 1] = acc\n    return Tflat, Marr, off, int(acc), Tstride          # acc == B(n)\n\n\ndef state_counts(n: int) -> Tuple[int, int]:\n    \"\"\"(B(n), 2*B(n)) -- row-boundary and mid-row peak state counts.\"\"\"\n    M = motzkin_numbers(n + 3)\n    B = M[n + 2] - M[n + 1]\n    return B, 2 * B\n\n\ndef bytes_needed(n: int) -> int:\n    \"\"\"Device bytes for the ping-pong pair of uint32 residue arrays.\"\"\"\n    return 2 * state_counts(n)[1] * 4\n\n\n# --------------------------------------------------------------------------\n# single-prime sweep on one GPU\n# --------------------------------------------------------------------------\nclass GpuSweep:\n    def __init__(self, n: int, device: int = 0, block: int = 256,\n                 grid: int = 4096) -> None:\n        import cupy as cp\n\n        self.cp, self.n, self.device = cp, n, device\n        self.block, self.grid = block, grid\n        with cp.cuda.Device(device):\n            mod = cp.RawModule(code=cuda_source(), backend=\"nvrtc\",\n                               options=(\"-std=c++11\", \"--use_fast_math\"))\n            self.k_step = mod.get_function(\"dp_step\")\n            self.k_rowend = mod.get_function(\"row_end\")\n            self.k_term = mod.get_function(\"terminal_sum\")\n\n            Tf, Mf, off, B, Tstride = build_tables(n)\n            self.B, self.Tstride = B, Tstride\n            self.size = 2 * B\n            self.dT = cp.asarray(Tf)\n            self.dM = cp.asarray(Mf)\n            self.dOff = cp.asarray(off)\n            self.shmem = (( n + 5) * Tstride + (n + 5) + (n + 2)) * 8\n            self.cur = cp.zeros(self.size, dtype=cp.uint32)\n            self.nxt = cp.zeros(self.size, dtype=cp.uint32)\n            self.acc = cp.zeros(1, dtype=cp.uint64)\n\n    def run(self, p: int, progress=None) -> int:\n        cp, n = self.cp, self.n\n        with cp.cuda.Device(self.device):\n            self.cur.fill(0)\n            # seed: vertex (0,0) emits the MARK down (idx 0) or right (idx 1)\n            self.cur[0:2] = 1\n            size_in = self.size\n            for i in range(n + 1):\n                for j in range(1 if i == 0 else 0, n + 1):\n                    from_boundary = 1 if j == 0 else 0\n                    if i == n and j == n:\n                        self.acc.fill(0)\n                        self.k_term((self.grid,), (self.block,),\n                                    (self.cur, self.acc, np.uint64(size_in),\n                                     np.int32(n), np.int32(from_boundary),\n                                     self.dT, self.dM, self.dOff,\n                                     np.int32(self.Tstride)),\n                                    shared_mem=self.shmem)\n                        return int(self.acc.get()[0] % p)\n                    self.nxt.fill(0)\n                    self.k_step((self.grid,), (self.block,),\n                                (self.cur, self.nxt, np.uint64(size_in),\n                                 np.int32(i), np.int32(j), np.int32(n),\n                                 np.uint32(p), np.int32(from_boundary),\n                                 self.dT, self.dM, self.dOff,\n                                 np.int32(self.Tstride)),\n                                shared_mem=self.shmem)\n                    self.cur, self.nxt = self.nxt, self.cur\n                    size_in = self.size\n                # row end -> boundary indexing, size B\n                self.nxt.fill(0)\n                self.k_rowend((self.grid,), (self.block,),\n                              (self.cur, self.nxt, np.uint64(self.B)))\n                self.cur, self.nxt = self.nxt, self.cur\n                size_in = self.B\n                if progress:\n                    progress(i + 1, n + 1)\n        raise RuntimeError(\"sweep finished without reaching the terminal vertex\")\n\n\n# --------------------------------------------------------------------------\n# CRT\n# --------------------------------------------------------------------------\ndef crt(residues: Sequence[int], primes: Sequence[int]) -> Tuple[int, int]:\n    total, N = 0, 1\n    for p in primes:\n        N *= p\n    for r, p in zip(residues, primes):\n        m = N // p\n        total = (total + r * m * pow(m, -1, p)) % N\n    return total, N\n\n\ndef estimate_bits(n: int) -> int:\n    \"\"\"log2 a(n) from the measured growth fit (629 bits at n=28).\"\"\"\n    return int(0.7479 * (n + 1) ** 2) + 8\n\n\ndef primes_for(n: int, margin: float = 1.30) -> List[int]:\n    need = int(estimate_bits(n) * margin)\n    out, bits = [], 0\n    for p in CRT_PRIMES_31BIT:\n        out.append(p)\n        bits += 30                      # each prime contributes > 2^30\n        if bits >= need:\n            break\n    return out\n\n\n# --------------------------------------------------------------------------\n# multi-GPU driver\n# --------------------------------------------------------------------------\ndef solve(n: int, primes: Sequence[int] | None = None,\n          devices: Sequence[int] | None = None, verbose: bool = True\n          ) -> Tuple[int, Dict[int, int], float]:\n    \"\"\"Exact a(n) via one full sweep per CRT prime, spread over the GPUs.\"\"\"\n    import cupy as cp\n\n    if devices is None:\n        devices = list(range(cp.cuda.runtime.getDeviceCount()))\n    if primes is None:\n        primes = primes_for(n)\n    residues: Dict[int, int] = {}\n    lock = threading.Lock()\n    t0 = time.time()\n\n    def worker(dev: int, my_primes: List[int]) -> None:\n        sweep = GpuSweep(n, device=dev)\n        for p in my_primes:\n            t1 = time.time()\n            r = sweep.run(p)\n            with lock:\n                residues[p] = r\n            if verbose:\n                print(f\"  [gpu{dev}] p={p}  a({n}) mod p = {r:>10d} \"\n                      f\"({time.time() - t1:.1f}s)\", flush=True)\n\n    buckets: List[List[int]] = [[] for _ in devices]\n    for k, p in enumerate(primes):\n        buckets[k % len(devices)].append(p)\n    threads = [threading.Thread(target=worker, args=(d, b))\n               for d, b in zip(devices, buckets) if b]\n    for t in threads:\n        t.start()\n    for t in threads:\n        t.join()\n\n    ordered = [residues[p] for p in primes]\n    value, _ = crt(ordered, list(primes))\n    return value, residues, time.time() - t0\n"}''')
for name, body in SOURCES.items():
    open('/kaggle/working/src/' + name, 'w').write(body)
import sys; sys.path.insert(0, '/kaggle/working/src')
print('wrote', list(SOURCES))

## 3. Verify the mathematics on CPU

Pure Python, no GPU. This proves the state-space theorem and the ranking bijection before any kernel runs.

In [ ]:
from a007764_core import (KNOWN_A007764, motzkin_numbers, ProfileRanker,
                          a_n_wordspace, a_n_dense, EMPTY, OPEN, CLOSE, MARK)
import itertools

M = motzkin_numbers(24)
for n in range(20):
    assert sum(M[a]*M[n-a] for a in range(n+1)) == M[n+2]-M[n+1]
print('B(n) = sum_a M_a M_(n-a) = M_(n+2) - M_(n+1)   verified n=0..19')

for n in range(0, 9):                       # exhaustive ranking round-trip
    P, seen = ProfileRanker(n), set()
    words = []
    for w in itertools.product([EMPTY, OPEN, CLOSE, MARK], repeat=n+1):
        if w.count(MARK) != 1:
            continue
        a = w.index(MARK)
        good = True
        for part in (w[:a], w[a+1:]):
            d = 0
            for c in part:
                d += (c == OPEN) - (c == CLOSE)
                if d < 0:
                    good = False
            if d != 0:
                good = False
        if good:
            words.append(list(w))
    assert len(words) == P.size == M[n+2]-M[n+1]
    for w in words:
        r = P.rank(w)
        assert 0 <= r < P.size and r not in seen and P.unrank(r) == w
        seen.add(r)
print('profile rank/unrank bijective onto [0,B(n))   verified exhaustively n=0..8')

for n in range(1, 10):
    assert a_n_wordspace(n) == KNOWN_A007764[n]
print('word-space frontier DP  == OEIS a(n)          verified n=1..9')
for n in range(1, 10):
    assert a_n_dense(n, report=True) == KNOWN_A007764[n]
print('dense rank-indexed DP   == OEIS a(n)          verified n=1..9, occupancy 1.0000')

## 4. Memory ledger

What actually fits, on this hardware and on a hypothetical 8×B300 node.

In [ ]:
from a007764_gpu import state_counts, bytes_needed, estimate_bits, primes_for
free0 = cp.cuda.Device(0).mem_info[0]
print(f'{"n":>3} {"B(n)":>18} {"peak = 2B(n)":>19} {"ping-pong uint32":>18}  fits one T4?')
best = None
for n in range(16, 30):
    B, pk = state_counts(n)
    need = bytes_needed(n)
    fits = need < free0 * 0.92
    if fits:
        best = n
    print(f'{n:>3} {B:>18,} {pk:>19,} {need/2**30:>15.2f} GiB  {"yes" if fits else "no"}')
print()
print(f'largest n that fits one T4 with this scheme: n = {best}')

## 5. GPU kernel against ground truth

The CUDA kernel must reproduce all twelve known terms modulo a prime before anything larger is attempted.

In [ ]:
from a007764_gpu import GpuSweep, CRT_PRIMES_31BIT
p = CRT_PRIMES_31BIT[1]
for n in range(1, 13):
    got = GpuSweep(n, device=0).run(p)
    exp = KNOWN_A007764[n] % p
    assert got == exp, (n, got, exp)
    print(f'  n={n:2d}: gpu = {got:>11d}  == a(n) mod p   OK')
print('CUDA kernel matches OEIS for every known term n=1..12')

## 6. Throughput ladder

Measure the real rate. Everything after this point is projected from these numbers, not assumed.

In [ ]:
import time
from a007764_gpu import state_counts
rates = {}
for n in [14, 16, 18]:
    B, pk = state_counts(n)
    t0 = time.time()
    GpuSweep(n, device=0).run(p)
    el = time.time() - t0
    cells = (n+1)**2
    rate = cells * pk / el
    rates[n] = rate
    print(f'  n={n:2d}: {el:7.2f}s for one prime   '
          f'{cells*pk:>16,} state-updates   {rate/1e9:6.3f} G updates/s')
rate = sum(rates.values()) / len(rates)
print(f'\nmean measured rate: {rate/1e9:.3f} G state-updates/s on one T4')

## 7. Main run

`TARGET_N` defaults to 20, which finishes comfortably. Set it to 21 to use the full T4 (10.9 GiB, the ceiling for this scheme); budget several hours.

In [ ]:
TARGET_N = 20        # 21 also fits one T4; 22 needs 30.7 GiB and does not

B, pk = state_counts(TARGET_N)
primes = primes_for(TARGET_N)
est = (TARGET_N+1)**2 * pk / rate * len(primes) / max(1, ndev)
print(f'n={TARGET_N}: {pk:,} states, {bytes_needed(TARGET_N)/2**30:.2f} GiB, '
      f'{len(primes)} primes')
print(f'projected wall clock on {ndev} GPU(s): {est/3600:.2f} h')

In [ ]:
from a007764_gpu import solve, crt
cp.get_default_memory_pool().free_all_blocks()
value, residues, elapsed = solve(TARGET_N, primes=primes,
                                 devices=list(range(ndev)))
print(f'\na({TARGET_N}) = {value}')
print(f'{value.bit_length()} bits, {len(str(value))} digits, {elapsed/60:.1f} min')

## 8. Independent confirmation

Add one prime beyond the estimated need and reconstruct again. An unchanged value proves the result is exact provided `a(n)` is below the enlarged modulus, which the 30-bit margin makes safe.

In [ ]:
cp.get_default_memory_pool().free_all_blocks()
extra = CRT_PRIMES_31BIT[len(primes)]
r_extra = GpuSweep(TARGET_N, device=0).run(extra)
value2, _ = crt([residues[q] for q in primes] + [r_extra], list(primes) + [extra])
print('with one extra prime:', 'UNCHANGED -> value is exact'
      if value2 == value else f'CHANGED -> need more primes ({value2})')
if TARGET_N in KNOWN_A007764:
    print('vs embedded OEIS value:',
          'MATCH' if value == KNOWN_A007764[TARGET_N] else 'MISMATCH')

## 9. What n=28 would take

Measured rate, extrapolated. No claim beyond arithmetic on the numbers above.

In [ ]:
B28, pk28 = state_counts(28)
cells28 = 29**2
updates28 = cells28 * pk28
prime_hours = updates28 / rate / 3600
nprimes28 = len(primes_for(28))
print(f'n=28 peak states      : {pk28:,}')
print(f'  uint32 ping-pong    : {bytes_needed(28)/2**40:.2f} TiB')
print(f'  11-bit single buffer: {pk28*11/8/2**40:.2f} TiB')
print(f'  11-bit + T-Sigma/2  : {pk28*11/8/2/2**40:.2f} TiB   '
      f'(vs 1.97 TiB on an 8xB300 node)')
print(f'state-updates         : {updates28:,}')
print(f'at the rate measured  : {prime_hours:,.0f} GPU-hours per prime, '
      f'{nprimes28} primes -> {prime_hours*nprimes28:,.0f} GPU-hours total')
print()
print('So the memory budget closes only without ping-pong, and the compute')
print('budget is the part nobody in this repository has ever measured before.')